# GEO proof of concept: analysis of the Qwen run (P11)

Extra analyses of the 60 answers produced by `geo_poc_pipeline.ipynb` (Qwen3-4B-Instruct-2507, 12 GEO-bench test queries × 5 answers).
**No GPU and no model needed**: this notebook only reads the saved results file from Google Drive.

1. Uncited sentences: where they are, how concentrated they are, export for hand-labelling
2. Word Count with both denominators (all words vs cited words only), position, noise (D19)
3. Reproducibility check across library versions (D18)

## 0. Setup: load the saved run

In [1]:
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/geo_lab/runs"
OUT_PATH = f"{OUT_DIR}/poc_qwen3-4b-instruct-2507.jsonl"

import re
import numpy as np
import pandas as pd

# Same word counting as the pipeline notebook (section 6b), needed for the uncited-sentence lengths
CITATION_RE = re.compile(r"\[(\d+(?:\s*,\s*\d+)*)\]")
def count_words(sentence):
    """Words in a sentence, ignoring citation markers and markdown symbols."""
    text = CITATION_RE.sub(" ", sentence)
    text = re.sub(r"[*#`]", "", text)
    return len([w for w in text.split() if re.search(r"\w", w)])

df = pd.read_json(OUT_PATH, lines=True)
print("answers:", len(df), "| queries:", df["row_idx"].nunique(),
      "| environments:", df["environment"].astype(str).unique().tolist())

Mounted at /content/drive
answers: 60 | queries: 12 | environments: ["{'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'gpu': 'Tesla T4'}"]


## 1. Uncited sentences
Hypothesis tested: most uncited sentences are introductions or summaries of the cited content.

### 1a. Where are they?

In [2]:
rows = []
for _, rec in df.iterrows():
    n = len(rec["parsed"])
    for i, s in enumerate(rec["parsed"]):
        if not s["cited"]:
            where = "first" if i == 0 else "last" if i == n - 1 else "middle"
            rows.append({"row_idx": rec["row_idx"], "sample": rec["sample"], "pos": i,
                         "n_sentences": n, "where": where,
                         "words": count_words(s["sentence"]), "sentence": s["sentence"]})
unc = pd.DataFrame(rows)

print(len(unc), "uncited sentences in", unc[["row_idx", "sample"]].drop_duplicates().shape[0], "answers")
print(unc["where"].value_counts())
for _, r in unc.iterrows():
    print(f"\n[row {r['row_idx']}, answer {r['sample']}, sentence {r['pos']+1}/{r['n_sentences']}, {r['where']}]  {r['sentence']}")

106 uncited sentences in 36 answers
where
middle    77
first     26
last       3
Name: count, dtype: int64

[row 933, answer 3, sentence 1/15, first]  King Edward of England refers to several historical monarchs, each with distinct reigns and legacies.

[row 933, answer 3, sentence 3/15, middle]  His reign saw the establishment of a strong central government and the expansion of English law and administrative systems.

[row 933, answer 3, sentence 9/15, middle]  Edward VI, born in 1537, was the son of Henry VIII and the last Tudor monarch to be born.

[row 933, answer 3, sentence 13/15, middle]  No other monarch named Edward is directly referenced in the provided results as a central figure of English history in the same way.

[row 933, answer 4, sentence 1/25, first]  The user's query refers to "King Edward of England," a title held by multiple English monarchs.

[row 933, answer 4, sentence 2/25, middle]  Based on the provided search results, the most historically significant and dir

### 1b. How concentrated are they? (and export for hand-labelling)

In [3]:
# How concentrated are uncited sentences? (per query, out of 5 answers)
per_query = unc.groupby("row_idx").agg(uncited_sentences=("sentence", "size"),
                                       answers_affected=("sample", "nunique"))
print(per_query.sort_values("uncited_sentences", ascending=False))

# Export for hand-labelling in Google Sheets
unc["label"] = ""   # fill in: intro / meta / conclusion / heading / claim
unc.to_csv(f"{OUT_DIR}/uncited_sentences_to_label.csv", index=False)
print("saved to", f"{OUT_DIR}/uncited_sentences_to_label.csv")

         uncited_sentences  answers_affected
row_idx                                     
933                     28                 5
916                     21                 5
856                     14                 5
972                     10                 5
127                      9                 4
608                      8                 4
635                      7                 2
147                      3                 1
260                      3                 2
859                      3                 3
saved to /content/drive/MyDrive/geo_lab/runs/uncited_sentences_to_label.csv


## 2. Word Count: all words vs cited words only (D19)
The two versions are linked exactly: WC(cited only) = WC(all words) ÷ (1 − uncited share). This section also looks at visibility by source position and at the noise between the 5 answers of a query.

In [4]:
# One row per (answer, source): both normalisations side by side
rows = []
for _, rec in df.iterrows():
    for i in range(5):
        wc_all = rec["word_count"][i]
        cited_share = 1 - rec["uncited_share"]
        rows.append({"row_idx": rec["row_idx"], "sample": rec["sample"], "source": i + 1,
                     "wc_all_words": wc_all,
                     "wc_cited_only": wc_all / cited_share if cited_share > 0 else np.nan,
                     "pawc": rec["position_adjusted_word_count"][i]})
src = pd.DataFrame(rows)

# Check: the two versions differ only by the factor (1 - uncited share)
check = df.apply(lambda r: abs(sum(r["word_count"]) - (1 - r["uncited_share"])), axis=1).max()
print(f"max |sum of WC - cited share| = {check:.2e}   (should be ~0)")

# 1. Average visibility by source position (1 = first source in the prompt)
print("\nMean visibility by source position (60 answers):")
print(src.groupby("source")[["wc_all_words", "wc_cited_only", "pawc"]].mean().round(3))

# 2. How noisy is a single answer? Std across the 5 answers of the same (query, source)
spread = src.groupby(["row_idx", "source"])["wc_all_words"].agg(["mean", "std"])
print(f"\nWord Count: mean over (query, source) = {spread['mean'].mean():.3f}, "
      f"typical std across the 5 answers = {spread['std'].mean():.3f}")

# 3. Does the choice of denominator change which source wins each query?
means = src.groupby(["row_idx", "source"])[["wc_all_words", "wc_cited_only"]].mean()
def winners(col):   # set of top sources per query (ties kept together)
    return means[col].groupby("row_idx").apply(
        lambda s: frozenset(s.index.get_level_values("source")[s >= s.max() - 1e-9]))
same = (winners("wc_all_words") == winners("wc_cited_only")).sum()
print(f"\nMost visible source per query is the same under both versions for {same} of {means.index.get_level_values('row_idx').nunique()} queries")

max |sum of WC - cited share| = 4.00e-15   (should be ~0)

Mean visibility by source position (60 answers):
        wc_all_words  wc_cited_only   pawc
source                                    
1              0.181          0.198  0.129
2              0.085          0.101  0.056
3              0.267          0.304  0.169
4              0.157          0.171  0.106
5              0.205          0.226  0.127

Word Count: mean over (query, source) = 0.179, typical std across the 5 answers = 0.052

Most visible source per query is the same under both versions for 12 of 12 queries


## 3. Reproducibility across library versions (D18)
Colab updated torch/transformers mid-run (2.11.0/5.22.0 → 2.14.1/5.22.1). The 8 answers made with the old versions were regenerated with the same seeds. Are they identical?

In [5]:
backup = pd.read_json(OUT_PATH.replace(".jsonl", "_backup_mixed_env.jsonl"), lines=True)
old = backup[backup["environment"].astype(str).str.contains("2.11.0")]

for _, o in old.iterrows():
    new = df[(df["row_idx"] == o["row_idx"]) & (df["sample"] == o["sample"])].iloc[0]
    print(f"row {o['row_idx']}, answer {o['sample']}: identical = {o['answer'] == new['answer']}")

row 978, answer 0: identical = True
row 978, answer 1: identical = True
row 978, answer 2: identical = True
row 978, answer 3: identical = True
row 978, answer 4: identical = True
row 933, answer 0: identical = True
row 933, answer 1: identical = True
row 933, answer 2: identical = True
